### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>

### Features
Batch run folder structure + manifest generator.
- Each run = one scenario-parameter combination (sequential R-number)
- Each seed = one BehaviorSpace run of that combination.

### Prerequisites

### Usage 
- python make_runs_R125.py                # create folders + CSV in ./
- python make_runs_R125.py --root PATH    # repo root (default: current dir)
- python make_runs_R125.py --dry-run       # show what would be created

## Roadmap:

### Version 1


# imports

In [ ]:
import os
import rasterio
import geopandas as gpd

import argparse
import csv
import hashlib
import os
from datetime import date
from pathlib import Path
import shutil

from ti_city_00_helper_functions import *
from ti_city_10_calibration_fuzzy_fom_validation_module import *

# init

In [ ]:
case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
git_path = Path(r'C:\\GIT\\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\\GIT\\ti-city-model' or 'C:\\Users\\x\\GIT\\ti-city-model'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_model_path = Path(f'../model/{case_city}/')
ti_city_in_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2000,2025]
    simulation_years = [2025, 2050]

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# Defs

In [10]:
# ----------------------------------------------------------------------------
# Batch definition — R125+ seed-controlled calibration batch (v41 model,
# post-R108 code, corrected inputs: all-roads layer + year-2000 bu proximity)
# ----------------------------------------------------------------------------

seeds_list = [11, 22, 33]
first_run = 125

# (R-offset, scenario name in scenarios.csv, purpose, parameter group variation)
# Baseline weight files put weight 10 on the single associated parameter for
# EVERY agent class (dp, li, mi, hi, mi_red, hi_red), all other weights 0.
runs_list = [
    {
        "offset": 0,
        "scenario": "s0",
        "purpose": "Calibration scenario (expert weight set)",
        "variation": "Full mechanism set — expert-informed point weights, unchanged",
    },
    {
        "offset": 1,
        "scenario": "s_random",
        "purpose": "Null baseline: random allocation",
        "variation": "random weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 2,
        "scenario": "s_edge",
        "purpose": "Null baseline: edge growth",
        "variation": "bu weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 3,
        "scenario": "s_roadbound",
        "purpose": "Null baseline: road-bound growth",
        "variation": "road weight = 10 (all 6 agent rows), all other weights 0",
    },
]

code_version = "v41 (post-R108 allocation fixes)"
inputs = "all-roads layer + bu-proximity year 2000 (corrected vintage)"
model_mode = "calibrate? = TRUE, RED-on?, neighbour-on? on"
hh_number = "frozen (gross target ~29,559; per scenarios.csv)"
status = "pending"


create_folders(
    seeds_list, 
    runs_list, 
    first_run, run_type="dry-run", 
    root_folder_path = git_path,
    inputs=inputs,
    code_version=code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status
)

Batch: 4 runs x 3 seeds = 12 BehaviorSpace runs
  R125  s0           Calibration scenario (expert weight set)
  R126  s_random     Null baseline: random allocation
  R127  s_edge       Null baseline: edge growth
  R128  s_roadbound  Null baseline: road-bound growth
C:\GIT\ti-city-model\model\None\data\out\R125\seed11
C:\GIT\ti-city-model\model\None\data\out\R125\seed22
C:\GIT\ti-city-model\model\None\data\out\R125\seed33
C:\GIT\ti-city-model\model\None\data\out\R126\seed11
C:\GIT\ti-city-model\model\None\data\out\R126\seed22
C:\GIT\ti-city-model\model\None\data\out\R126\seed33
C:\GIT\ti-city-model\model\None\data\out\R127\seed11
C:\GIT\ti-city-model\model\None\data\out\R127\seed22
C:\GIT\ti-city-model\model\None\data\out\R127\seed33
C:\GIT\ti-city-model\model\None\data\out\R128\seed11
C:\GIT\ti-city-model\model\None\data\out\R128\seed22
C:\GIT\ti-city-model\model\None\data\out\R128\seed33

Manifest would be written to: C:\GIT\ti-city-model\docs\None_TA-model_manifest_R125_128.csv

(dry

In [11]:
code_version = "v41 (post-R108 allocation fixes)"
inputs = "all-roads layer + bu-proximity year 2000 (corrected vintage)"
model_mode = "calibrate? = TRUE, RED-on?, neighbour-on? on"
hh_number = "frozen (gross target ~29,559; per scenarios.csv)"
status = "pending"
runs_list = [
    {
        "offset": 0,
        "scenario": "s0",
        "purpose": "Calibration scenario (expert weight set)",
        "variation": "Full mechanism set — expert-informed point weights, unchanged",
    },
    {
        "offset": 1,
        "scenario": "s_random",
        "purpose": "Null baseline: random allocation",
        "variation": "random weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 2,
        "scenario": "s_edge",
        "purpose": "Null baseline: edge growth",
        "variation": "bu weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 3,
        "scenario": "s_roadbound",
        "purpose": "Null baseline: road-bound growth",
        "variation": "road weight = 10 (all 6 agent rows), all other weights 0",
    },
]
create_folders(
    case_city=case_city,
    seeds_list = seeds_list, 
    runs_list = runs_list, 
    first_run = first_run, 
    run_type = "create", 
    root_folder_path = git_path,
    inputs = inputs,
    code_version = code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status,
)

TypeError: create_folders() got an unexpected keyword argument 'case_city'

# The end (for now).